Import libraries

In [4]:
#import
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader
from cfrnet import CFRnet
import sys
from pathlib import Path
project_root = Path("/home/ducm/Benchmark-conversion-vs-revenue-uplift-modeling")
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))
from utils import seed_everything
from metrics import auuc, auqc, lift, krcc

In [5]:
%time train_df = pd.read_csv(r"../../dataset/Hillstrom/Women/train_women.csv")
%time test_df =  pd.read_csv(r"../../dataset/Hillstrom/Women/test_women.csv")
%time val_df = pd.read_csv(r"../../dataset/Hillstrom/Women/val_women.csv")

CPU times: user 24.5 ms, sys: 5.97 ms, total: 30.5 ms
Wall time: 30.2 ms
CPU times: user 14.2 ms, sys: 1.01 ms, total: 15.2 ms
Wall time: 15.2 ms
CPU times: user 4.28 ms, sys: 0 ns, total: 4.28 ms
Wall time: 4.28 ms


In [6]:
in_features = ['recency', 'history_segment', 'history', 'mens', 'womens',
       'zip_code', 'newbie', 'channel_Multichannel', 'channel_Phone', 'channel_Web']
label_feature = ['spend']
treatment_feature = ['treatment']

In [7]:
X_train = train_df[in_features].values.astype(float) # type: ignore
y_train = train_df[label_feature].values.astype(float) # type: ignore
t_train = train_df[treatment_feature].values.astype(float) # type: ignore

X_test = test_df[in_features].values.astype(float) # type: ignore
y_test = test_df[label_feature].values.astype(float) # type: ignore
t_test = test_df[treatment_feature].values.astype(float) # type: ignore

X_val = val_df[in_features].values.astype(float) # type: ignore
y_val = val_df[label_feature].values.astype(float) # type: ignore
t_val = val_df[treatment_feature].values.astype(float) # type: ignore

In [8]:
print('X_train[:10]', X_train[:1].astype(float))

X_train[:10] [[-0.78953321  1.6460815   1.41745802 -1.11692492  0.91314538  1.07643872
   0.9950542  -0.36894675 -0.88924565  1.13133938]]


In [9]:
print('y_train[:10]', y_train[:1].astype(float))

y_train[:10] [[0.]]


In [10]:
# Transform to tensor
def to_tensor(arr):
    return torch.tensor(arr, dtype=torch.float32)

x_men_train_t = to_tensor(X_train)
x_men_val_t = to_tensor(X_val)
x_men_test_t = to_tensor(X_test)

y_men_train_t = to_tensor(y_train).reshape(-1, 1)
y_men_val_t = to_tensor(y_val).reshape(-1, 1)
y_men_test_t = to_tensor(y_test).reshape(-1, 1)

# t_train/t_val/t_test cũng tương tự
t_men_train_t = to_tensor(t_train.astype(float)).reshape(-1, 1)
t_men_val_t = to_tensor(t_val.astype(float)).reshape(-1, 1)
t_men_test_t = to_tensor(t_test.astype(float)).reshape(-1, 1)

# Data loader
train_dataset = TensorDataset(x_men_train_t, t_men_train_t, y_men_train_t)
val_dataset = TensorDataset(x_men_val_t, t_men_val_t, y_men_val_t)
test_dataset = TensorDataset(x_men_test_t, t_men_test_t, y_men_test_t)

batch_size = 6400
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory = True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory = True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory = True)

print("-------------------------------------------------------------")
print("✅ Completed transform to tensor ✅")
print(f"Shape of train: x={x_men_train_t.shape}; y={y_men_train_t.shape}; t={t_men_train_t.shape}")
print(f"Shape of val: x={x_men_val_t.shape}; y={y_men_val_t.shape}; t={t_men_val_t.shape}")
print(f"Shape of test: x={x_men_test_t.shape}; y={y_men_test_t.shape}; t={t_men_test_t.shape}")

-------------------------------------------------------------
✅ Completed transform to tensor ✅
Shape of train: x=torch.Size([25615, 10]); y=torch.Size([25615, 1]); t=torch.Size([25615, 1])
Shape of val: x=torch.Size([4270, 10]); y=torch.Size([4270, 1]); t=torch.Size([4270, 1])
Shape of test: x=torch.Size([12808, 10]); y=torch.Size([12808, 1]); t=torch.Size([12808, 1])


In [11]:
epochs = 150
alpha = 0.1
lr = 1e-3
wd = 1e-5
method = "mmd_rbf"
early_stop_metric = "qini"
ema = True
ema_alpha = 0.15
patience = 20
early_stop_start = 30
shared_hidden = 200
outcome_hidden = 100
outcome_dropout = 0
shared_dropout = 0.0
activation = torch.nn.ReLU

print (f" epochs = {epochs}")
print (f" method = {method}")
print (f" alpha = {alpha}")
print (f" learning rate = {lr}")
print (f" weight decay = {wd}")
print (f" early stop = {early_stop_metric}")
print (f" use ema = {ema}")
print (f" ema alpha = {ema_alpha}")
print (f" patience = {patience}")
print (f" shared hidden = {shared_hidden}")
print (f" outcome hidden = {outcome_hidden}")
print (f"activation = {activation}")

 epochs = 150
 method = mmd_rbf
 alpha = 0.1
 learning rate = 0.001
 weight decay = 1e-05
 early stop = qini
 use ema = True
 ema alpha = 0.15
 patience = 20
 shared hidden = 200
 outcome hidden = 100
activation = <class 'torch.nn.modules.activation.ReLU'>


In [12]:
import pandas as pd
import numpy as np
import torch

# 1. Cấu hình
seeds = [412312, 42, 1874, 902745, 1]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
all_runs = []

print(f"🔄 Đang huấn luyện TARNet trên {len(seeds)} seeds khác nhau... Vui lòng đợi.")

# 2. Vòng lặp chạy (Chỉ xử lý, không in kết quả lẻ)
for SEED in seeds:
    seed_everything(SEED)
    
    # Khởi tạo lại mô hình
    cfr = CFRnet(
        input_dim=x_men_train_t.shape[1], epochs=epochs, learning_rate=lr, 
        alpha = alpha, method=method,
        weight_decay=wd, use_ema=ema, ema_alpha=ema_alpha, patience=patience,
        shared_hidden=shared_hidden, outcome_hidden=outcome_hidden,
        outcome_dropout=outcome_dropout, shared_dropout=shared_dropout,
        early_stop_metric=early_stop_metric, early_stop_start_epoch=early_stop_start,
        activation=activation
    )
    
    cfr.fit(train_loader, val_loader)
    
    # Predict
    x_men_test_t_on_device = x_men_test_t.to(device)
    y0_pred, y1_pred = cfr.predict(x_men_test_t_on_device)
    
    uplift_pred = (y1_pred - y0_pred).cpu().numpy().flatten()
    y_true = y_men_test_t.cpu().numpy().flatten()
    t_true = t_men_test_t.cpu().numpy().flatten()
    
    # Tính toán
    ate_pred = uplift_pred.mean()
    ate_true = y_true[t_true == 1].mean() - y_true[t_true == 0].mean()
    
    all_runs.append({
        'Seed': SEED,
        'AUUC': auuc(y_true, t_true, uplift_pred, bins=100, plot=False),
        'AUQC': auqc(y_true, t_true, uplift_pred, bins=100, plot=False),
        'Lift': lift(y_true, t_true, uplift_pred, h=0.3),
        'KRCC': krcc(y_true, t_true, uplift_pred, bins=100),
        'ATE_Err': abs(ate_pred - ate_true)
    })
    print(f"  ✔️ Đã xong Seed {SEED}")

# 3. HIỂN THỊ KẾT QUẢ TỔNG HỢP (Print 1 lúc)
df_results = pd.DataFrame(all_runs)

print("\n" + "="*85)
print(f"{'CHI TIẾT TỪNG SEED':^85}")
print("="*85)
# Sử dụng to_string để in bảng đẹp mắt
print(df_results.to_string(index=False, formatters={
    'AUUC': '{:,.4f}'.format, 'AUQC': '{:,.4f}'.format, 
    'Lift': '{:,.4f}'.format, 'KRCC': '{:,.4f}'.format, 'ATE_Err': '{:,.4f}'.format
}))

# 4. Tính toán Mean và Std
mean_res = df_results.drop(columns='Seed').mean()
std_res = df_results.drop(columns='Seed').std()

print("="*85)
print(f"{'KẾT QUẢ TRUNG BÌNH (MEAN ± STD)':^85}")
print("-" * 85)
summary_data = []
for metric in ['AUUC', 'AUQC', 'Lift', 'KRCC', 'ATE_Err']:
    print(f"{metric:<10}: {mean_res[metric]:.4f} ± {std_res[metric]:.4f}")

print("="*85)

🔄 Đang huấn luyện TARNet trên 5 seeds khác nhau... Vui lòng đợi.
Locked random seed: 412312
🔃🔃🔃Begin training Dragonnet🔃🔃🔃
📊 Early Stop Metric: QINI
📊 Early Stop Start Epoch: 31
📊 Strategy: Two-Stage EMA Filter (alpha=0.15)
   EMA filters noise spikes, Raw Qini determines peak height
   Select checkpoint: raw_qini is highest AND raw_qini >= ema_qini
Epoch 1/150 | Loss: -0.6501 | Val Loss: 192.3704 | Val Qini: 0.2836 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.2836 | ⭐ NEW BEST (peak ≥ trend)
Epoch 2/150 | Loss: 459.4770 | Val Loss: 191.9535 | Val Qini: 0.9727 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.3870 | ⭐ NEW BEST (peak ≥ trend)
Epoch 3/150 | Loss: 3.6858 | Val Loss: 192.5904 | Val Qini: 1.1574 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.5026 | ⭐ NEW BEST (peak ≥ trend)
Epoch 4/150 | Loss: 1.2942 | Val Loss: 191.9818 | Val Qini: 0.9876 ✓ above trend but not peak (patience: 1/20)EMA Trend: 0.5753 | ✓ above trend but not peak (patience: 1/20)
Epoch 5/150 | Loss: 0.4340 | Val Loss: 191.9214 | Val Qin

/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/Revenue/CFRnet/cfrnet.py:276: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)


Epoch 1/150 | Loss: -0.7157 | Val Loss: 192.4474 | Val Qini: -0.0051 ⭐ NEW BEST (peak ≥ trend)EMA Trend: -0.0051 | ⭐ NEW BEST (peak ≥ trend)
Epoch 2/150 | Loss: -0.2225 | Val Loss: 191.9868 | Val Qini: 0.3072 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.0417 | ⭐ NEW BEST (peak ≥ trend)
Epoch 3/150 | Loss: 0.8304 | Val Loss: 192.0164 | Val Qini: 0.9709 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.1811 | ⭐ NEW BEST (peak ≥ trend)
Epoch 4/150 | Loss: 0.2402 | Val Loss: 191.8638 | Val Qini: 0.9854 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.3017 | ⭐ NEW BEST (peak ≥ trend)
Epoch 5/150 | Loss: 0.5964 | Val Loss: 191.8537 | Val Qini: 0.9849 ✓ above trend but not peak (patience: 1/20)EMA Trend: 0.4042 | ✓ above trend but not peak (patience: 1/20)
Epoch 6/150 | Loss: 261.4723 | Val Loss: 192.0448 | Val Qini: 1.0492 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.5009 | ⭐ NEW BEST (peak ≥ trend)
Epoch 7/150 | Loss: 1.9133 | Val Loss: 192.1769 | Val Qini: 1.0063 ✓ above trend but not peak (patience: 1/20)EMA Trend: 0.5767 | 

/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/Revenue/CFRnet/cfrnet.py:276: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)


Epoch 1/150 | Loss: -0.7540 | Val Loss: 192.6002 | Val Qini: 0.7375 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.7375 | ⭐ NEW BEST (peak ≥ trend)
Epoch 2/150 | Loss: -0.4184 | Val Loss: 192.0417 | Val Qini: 0.7942 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.7460 | ⭐ NEW BEST (peak ≥ trend)
Epoch 3/150 | Loss: 1.1652 | Val Loss: 192.1622 | Val Qini: 0.7384 (patience: 1/20)EMA Trend: 0.7448 | (patience: 1/20)
Epoch 4/150 | Loss: 1.6839 | Val Loss: 192.0503 | Val Qini: 0.7509 ✓ above trend but not peak (patience: 2/20)EMA Trend: 0.7457 | ✓ above trend but not peak (patience: 2/20)
Epoch 5/150 | Loss: 0.5725 | Val Loss: 191.8260 | Val Qini: 0.8038 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.7544 | ⭐ NEW BEST (peak ≥ trend)
Epoch 6/150 | Loss: 0.2150 | Val Loss: 191.7984 | Val Qini: 0.8811 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.7734 | ⭐ NEW BEST (peak ≥ trend)
Epoch 7/150 | Loss: 0.2920 | Val Loss: 191.8077 | Val Qini: 0.8771 ✓ above trend but not peak (patience: 1/20)EMA Trend: 0.7890 | ✓ above trend but not 

/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/Revenue/CFRnet/cfrnet.py:276: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)


Epoch 1/150 | Loss: -0.7238 | Val Loss: 192.5068 | Val Qini: 0.7964 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.7964 | ⭐ NEW BEST (peak ≥ trend)
Epoch 2/150 | Loss: -0.3758 | Val Loss: 192.0773 | Val Qini: 0.8151 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.7992 | ⭐ NEW BEST (peak ≥ trend)
Epoch 3/150 | Loss: 1.4120 | Val Loss: 192.2994 | Val Qini: 1.0535 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.8374 | ⭐ NEW BEST (peak ≥ trend)
Epoch 4/150 | Loss: 2.2019 | Val Loss: 192.2378 | Val Qini: 1.0745 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.8729 | ⭐ NEW BEST (peak ≥ trend)
Epoch 5/150 | Loss: 0.3588 | Val Loss: 191.8844 | Val Qini: 1.0554 ✓ above trend but not peak (patience: 1/20)EMA Trend: 0.9003 | ✓ above trend but not peak (patience: 1/20)
Epoch 6/150 | Loss: 0.3597 | Val Loss: 191.8769 | Val Qini: 1.0563 ✓ above trend but not peak (patience: 2/20)EMA Trend: 0.9237 | ✓ above trend but not peak (patience: 2/20)
Epoch 7/150 | Loss: 0.7990 | Val Loss: 191.9556 | Val Qini: 0.9915 ✓ above trend but not peak (pat

/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/Revenue/CFRnet/cfrnet.py:276: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)


Epoch 1/150 | Loss: -0.7477 | Val Loss: 192.5181 | Val Qini: 1.0500 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 1.0500 | ⭐ NEW BEST (peak ≥ trend)
Epoch 2/150 | Loss: -0.3799 | Val Loss: 191.9645 | Val Qini: 1.1786 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 1.0693 | ⭐ NEW BEST (peak ≥ trend)
Epoch 3/150 | Loss: 1.4318 | Val Loss: 192.1786 | Val Qini: 1.0327 (patience: 1/20)EMA Trend: 1.0638 | (patience: 1/20)
Epoch 4/150 | Loss: 0.8119 | Val Loss: 191.9632 | Val Qini: 0.9528 (patience: 2/20)EMA Trend: 1.0472 | (patience: 2/20)
Epoch 5/150 | Loss: 0.7831 | Val Loss: 191.8887 | Val Qini: 0.9803 (patience: 3/20)EMA Trend: 1.0371 | (patience: 3/20)
Epoch 6/150 | Loss: 1.3208 | Val Loss: 191.9801 | Val Qini: 0.9693 (patience: 4/20)EMA Trend: 1.0270 | (patience: 4/20)
Epoch 7/150 | Loss: 1525.1078 | Val Loss: 192.0854 | Val Qini: 0.8920 (patience: 5/20)EMA Trend: 1.0067 | (patience: 5/20)
Epoch 8/150 | Loss: 0.8379 | Val Loss: 192.1568 | Val Qini: 0.9052 (patience: 6/20)EMA Trend: 0.9915 | (patience: 6/20)

/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/Revenue/CFRnet/cfrnet.py:276: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)
